Importación del dataset


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import pandas as pd

path = "/content/drive/MyDrive/Data Science 3 TP/mbti_1.csv"
df = pd.read_csv(path)
print(f"Filas cargadas: {len(df)}")
print(df.head(4))


Filas cargadas: 8675
   type                                              posts
0  INFJ  'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1  ENTP  'I'm finding the lack of me in these posts ver...
2  INTP  'Good one  _____   https://www.youtube.com/wat...
3  INTJ  'Dear INTP,   I enjoyed our conversation the o...


Configuración del entorno

In [14]:
# Paso 1: En PyTorch siempre empezamos detectando si hay GPU, MPS (Apple Silicon) o CPU

# Importar librería pytorch
# Device torch→ Define el dispositivo de procesamiento (device) donde se ejecutarán tus tensores y modelos
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Usando dispositivo: {device}")


Usando dispositivo: cpu


In [15]:
# Paso 2: Definir el modelo (nn.Module)
# Creamos un perceptrón multicapa (MLP) base

import torch.nn as nn

class MBTIClassifier(nn.Module):

  def __init__(self, input_dim, hidden_dim=32, num_classes=2):
    super(MBTIClassifier, self).__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, num_classes),
    )

  def forward(self, x):
    return self.network(x)


In [16]:
# Paso 3: Preparar datos y DataLoader
# Convertimos los datos de entrenamiento y validación en tensores y los empaquetamos

from torch.utils.data import DataLoader, TensorDataset

# X_train, y_train, X_val, y_val deben ser tensores (torch.float32 y torch.long)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [17]:
#Paso 4 Loop de entrenamiento y validación con Adam

model = MBTIClassifier(input_dim=X_train_tensor.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 10
train_losses = []

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for features, labels in train_loader:
    features, labels = features.to(device), labels.to(device)

    optimizer.zero_grad()  # Limpiar gradientes
    outputs = model(features)  # Forward
    loss = criterion(outputs, labels)  # Calcular pérdida
    loss.backward()  # Backpropagation (torch.autograd)
    optimizer.step()  # Actualizar pesos

    running_loss += loss.item()

  avg_loss = running_loss / len(train_loader)
  train_losses.append(avg_loss)
  print(f"Época [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

# Evaluación en validación
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for features, labels in val_loader:
    features, labels = features.to(device), labels.to(device)
    outputs = model(features)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy en validación: {accuracy:.2f}%")


Época [1/10] - Loss: 0.5601
Época [2/10] - Loss: 0.5284
Época [3/10] - Loss: 0.5261
Época [4/10] - Loss: 0.5250
Época [5/10] - Loss: 0.5241
Época [6/10] - Loss: 0.5235
Época [7/10] - Loss: 0.5229
Época [8/10] - Loss: 0.5230
Época [9/10] - Loss: 0.5227
Época [10/10] - Loss: 0.5228
Accuracy en validación: 77.00%
